In [ ]:
import importlib.util, json, os, random, subprocess, sys
from pathlib import Path

_REQUIRED = {"pandas": "pandas", "pyarrow": "pyarrow"}
_missing = [package for module, package in _REQUIRED.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

import numpy as np
import pandas as pd

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find shared config.json from the notebook working directory")
CONFIG = json.loads((ROOT / "config.json").read_text(encoding="utf-8"))
SEED = int(CONFIG["random_seed"])
random.seed(SEED)
np.random.seed(SEED)
for key in ("data_dir", "artifact_dir", "results_dir"):
    (ROOT / CONFIG[key]).mkdir(parents=True, exist_ok=True)
ARTIFACTS = ROOT / CONFIG["artifact_dir"]
RESULTS = ROOT / CONFIG["results_dir"]
print(f"Setup complete; smoke_test_mode={CONFIG['smoke_test_mode']}; seed={SEED}")

# 01 — Data, catalogue, labels, and splits

Exports the catalogue and synthetic queries directly from the v3 reference notebook. Catalogue verification claims are preserved as `claimed_verification`; no policy is marked verified. Synthetic queries have no synthetic relevance labels.

In [ ]:
import ast, csv, itertools, math

source_path = ROOT / CONFIG["source_notebook"]
if not source_path.is_file():
    raise FileNotFoundError(f"Reference notebook not found: {source_path}")
source_nb = json.loads(source_path.read_text(encoding="utf-8"))
code_cells = ["".join(c.get("source", [])) for c in source_nb["cells"] if c.get("cell_type") == "code"]

def find_source_cell(marker):
    return next((source for source in code_cells if marker in source), None)

def assignment_value(source, name):
    tree = ast.parse(source)
    for node in ast.walk(tree):
        if isinstance(node, ast.AnnAssign) and isinstance(node.target, ast.Name) and node.target.id == name:
            return ast.literal_eval(node.value)
        if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == name for t in node.targets):
            return ast.literal_eval(node.value)
    raise ValueError(f"Could not extract literal {name} from the reference notebook")

catalogue_source = find_source_cell("AUDITED_LIC_CATALOGUE")
if catalogue_source is None:
    raise ValueError("Reference notebook has no AUDITED_LIC_CATALOGUE")
policies = assignment_value(catalogue_source, "AUDITED_LIC_CATALOGUE")
for policy in policies:
    policy["claimed_verification"] = policy.pop("verification_status", "")
    policy["verified"] = False
    policy["source_note"] = ""

additions_path = ROOT / "data" / "policy_additions.csv"
if additions_path.is_file():
    additions = pd.read_csv(additions_path, dtype=str).fillna("")
    required_addition_columns = {"policy_id", "policy_name", "plan_number", "uin", "source_note"}
    missing_addition_columns = required_addition_columns - set(additions.columns)
    if missing_addition_columns:
        raise ValueError(f"Policy additions are missing columns: {sorted(missing_addition_columns)}")
    policy_fields = set(policies[0])
    for addition in additions.to_dict(orient="records"):
        if addition["policy_id"] in {policy["policy_id"] for policy in policies}:
            raise ValueError(f"Duplicate policy ID in additions: {addition['policy_id']}")
        policy = {field: None for field in policy_fields}
        policy.update({
            "policy_id": addition["policy_id"],
            "policy_name": addition["policy_name"],
            "plan_number": addition["plan_number"] or None,
            "uin": addition["uin"] or None,
            "claimed_verification": "",
            "verified": False,
            "source_note": addition["source_note"],
        })
        policies.append(policy)

persona_source = find_source_cell("personas = [")
if persona_source is None:
    raise ValueError("Reference notebook has no literal synthetic persona list")
personas = assignment_value(persona_source, "personas")
queries = []
for persona in personas:
    for query in persona["templates"]:
        query_id = f"SYN_Q_{len(queries) + 1:03d}"
        queries.append({
            "query_id": query_id,
            "group_id": query_id,
            "raw_query": query,
            "persona": persona["persona"],
            "provenance": "SYNTHETIC",
        })

policies_df = pd.DataFrame(policies)
queries_df = pd.DataFrame(queries)
if policies_df["policy_id"].duplicated().any() or queries_df["query_id"].duplicated().any():
    raise ValueError("Duplicate policy or query IDs in source notebook")
known_age_limits = policies_df["min_age"].notna() & policies_df["max_age"].notna()
if not (policies_df.loc[known_age_limits, "min_age"] <= policies_df.loc[known_age_limits, "max_age"]).all():
    raise ValueError("Catalogue contains an invalid entry-age interval")
known_term_limits = policies_df["min_term"].notna() & policies_df["max_term"].notna()
if not (policies_df.loc[known_term_limits, "min_term"] <= policies_df.loc[known_term_limits, "max_term"]).all():
    raise ValueError("Catalogue contains an invalid policy-term interval")

(ROOT / "data").mkdir(exist_ok=True)
policies_df.to_csv(ROOT / "data" / "policies.csv", index=False, quoting=csv.QUOTE_MINIMAL)
policies_df.to_parquet(ARTIFACTS / "policies.parquet", index=False)
queries_df.to_csv(ARTIFACTS / "queries.csv", index=False)

# Only the explicit labels/relevance_labels.csv path is considered human-labeled data.
real_path = ROOT / CONFIG["real_labels_path"]
template_candidates = [ROOT / CONFIG["labels_template_path"], ROOT / "labels" / "relevance_labels_TEMPLATE.csv"]
template_path = next((p for p in template_candidates if p.is_file()), None)
label_columns = ["query_id", "group_id", "policy_id", "grade", "annotator"]
labels_df = pd.DataFrame(columns=label_columns)
label_source = None
if real_path.is_file():
    raw_labels = pd.read_csv(real_path)
    missing = set(label_columns) - set(raw_labels.columns)
    if missing:
        raise ValueError(f"Human labels are missing columns: {sorted(missing)}")
    raw_labels["grade"] = pd.to_numeric(raw_labels["grade"], errors="coerce")
    labels_df = raw_labels.loc[raw_labels["grade"].notna(), label_columns].copy()
    if not labels_df.empty:
        query_groups = dict(zip(queries_df["query_id"], queries_df["group_id"]))
        if not set(labels_df["query_id"]).issubset(query_groups):
            raise ValueError("Human labels contain query IDs absent from the exported query set")
        if not set(labels_df["policy_id"]).issubset(set(policies_df["policy_id"])):
            raise ValueError("Human labels contain policy IDs absent from the catalogue")
        if any(query_groups[qid] != group_id for qid, group_id in zip(labels_df["query_id"], labels_df["group_id"])):
            raise ValueError("Human label group_id does not match its query's split group")
        if labels_df.duplicated(["query_id", "policy_id", "annotator"]).any():
            raise ValueError("Duplicate human label from the same annotator for a query-policy pair")
    if not labels_df["grade"].between(0, 3).all() or not (labels_df["grade"] % 1 == 0).all():
        raise ValueError("Human relevance grades must be integers from 0 to 3")
    label_source = str(real_path.relative_to(ROOT))
labels_df.to_parquet(ARTIFACTS / "relevance_labels.parquet", index=False)

def cohens_kappa(a, b):
    observed = sum(x == y for x, y in zip(a, b)) / len(a)
    categories = sorted(set(a) | set(b))
    expected = sum((a.count(c) / len(a)) * (b.count(c) / len(b)) for c in categories)
    return (1.0 if observed == 1.0 else None) if math.isclose(expected, 1.0) else (observed - expected) / (1.0 - expected)

kappa_pairs = []
if not labels_df.empty:
    labels_df["annotator"] = labels_df["annotator"].fillna("").astype(str)
    annotators = sorted(a for a in labels_df["annotator"].unique() if a)
    for first, second in itertools.combinations(annotators, 2):
        a = labels_df[labels_df["annotator"] == first].set_index(["query_id", "policy_id"])["grade"]
        b = labels_df[labels_df["annotator"] == second].set_index(["query_id", "policy_id"])["grade"]
        common = a.index.intersection(b.index)
        if len(common):
            av, bv = [int(a.loc[key]) for key in common], [int(b.loc[key]) for key in common]
            kappa_pairs.append({"annotator_1": first, "annotator_2": second, "n_overlap": len(common), "cohens_kappa": cohens_kappa(av, bv)})
else:
    annotators = []

# Split groups, not rows, so a query and its paraphrases can never leak across splits.
groups = sorted(queries_df["group_id"].unique().tolist())
random.Random(SEED).shuffle(groups)
n_test = max(1, round(0.20 * len(groups)))
n_validation = max(1, round(0.20 * len(groups)))
if n_test + n_validation >= len(groups):
    raise ValueError("Not enough query groups for train/validation/test splits")
group_split = {
    "train": groups[:len(groups) - n_validation - n_test],
    "validation": groups[len(groups) - n_validation - n_test:len(groups) - n_test],
    "test": groups[len(groups) - n_test:],
}
query_split = {name: queries_df.loc[queries_df.group_id.isin(ids), "query_id"].tolist() for name, ids in group_split.items()}
if any(set(group_split[a]) & set(group_split[b]) for a, b in itertools.combinations(group_split, 2)):
    raise AssertionError("A query group appears in more than one split")
policy_ids = sorted(policies_df["policy_id"].tolist())
random.Random(SEED + 1).shuffle(policy_ids)
n_policy_test = max(1, round(0.20 * len(policy_ids)))
splits = {
    "seed": SEED,
    "group_split": {"groups": group_split, "query_ids": query_split},
    "policy_held_out": {"training_policy_ids": sorted(policy_ids[n_policy_test:]), "held_out_policy_ids": sorted(policy_ids[:n_policy_test])},
}
(ARTIFACTS / "splits.json").write_text(json.dumps(splits, indent=2), encoding="utf-8")

annotator_values = []
if template_path is not None:
    template = pd.read_csv(template_path)
    graded = template.loc[template["grade"].notna()] if "grade" in template else template.iloc[0:0]
    template_grade_count = int(len(graded))
    annotator_values = sorted(str(a) for a in graded["annotator"].dropna().unique()) if "annotator" in graded else []
else:
    template_grade_count = 0

status = {
    "mode": "human_labels" if len(labels_df) else "smoke_test_only",
    "catalogue_rows_exported": int(len(policies_df)),
    "synthetic_queries_exported": int(len(queries_df)),
    "query_groups": int(len(groups)),
    "human_label_source": label_source,
    "human_graded_rows": int(len(labels_df)),
    "human_annotators": annotators,
    "template_path": str(template_path.relative_to(ROOT)) if template_path else None,
    "template_rows": int(len(template)) if template_path is not None else 0,
    "template_graded_rows": template_grade_count,
    "template_annotators_with_grades": annotator_values,
    "cohens_kappa": {"status": "computed" if kappa_pairs else "not_computed_less_than_two_annotators_with_overlapping_grades", "pairs": kappa_pairs},
    "synthetic_labels_used_for_evaluation": False,
    "group_split_sizes": {key: int(len(value)) for key, value in query_split.items()},
}
(RESULTS / "data_status.json").write_text(json.dumps(status, indent=2), encoding="utf-8")
print(json.dumps(status, indent=2))

## Assertions

In [ ]:
assert len(policies_df) == 47
assert len(queries_df) == 18
assert policies_df["verified"].eq(False).all()
assert policies_df["claimed_verification"].notna().all()
additions = policies_df[policies_df["source_note"].fillna("").ne("")]
assert len(additions) == 27
assert additions[["min_age", "max_age", "min_term", "max_term"]].isna().all().all()
assert queries_df["provenance"].eq("SYNTHETIC").all()
assert "expected_goals" not in queries_df.columns
all_group_sets = [set(group_split[name]) for name in ("train", "validation", "test")]
assert all(not (a & b) for i, a in enumerate(all_group_sets) for b in all_group_sets[i + 1:])
assert set().union(*all_group_sets) == set(queries_df["group_id"])
assert set(splits["policy_held_out"]["training_policy_ids"]).isdisjoint(splits["policy_held_out"]["held_out_policy_ids"])
print("Notebook 01 checks passed: catalogue provenance, synthetic query export, and group-disjoint splits.")